<a href="https://colab.research.google.com/github/kabir-codes/ml-challenge/blob/jigsaw-dragon/Ml_Challenge_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [198]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MultiLabelBinarizer
import re
from sklearn.model_selection import train_test_split

In [199]:
df = pd.read_csv("/content/ml_challenge_dataset.csv")
df

,unique_id,Painting,"On a scale of 1–10, how intense is the emotion conveyed by the artwork?",Describe how this painting makes you feel.,This art piece makes me feel sombre.,This art piece makes me feel content.,This art piece makes me feel calm.,This art piece makes me feel uneasy.,How many prominent colours do you notice in this painting?,How many objects caught your eye in the painting?,How much (in Canadian dollars) would you be willing to pay for this painting?,"If you could purchase this painting, which room would you put that painting in?","If you could view this art in person, who would you want to view it with?",What season does this art piece remind you of?,"If this painting was a food, what would be?",Imagine a soundtrack for this painting. Describe that soundtrack without naming any objects in the painting.
0,1,The Persistence of Memory,NaN,NaN,NaN,NaN,NaN,NaN,5.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,The Persistence of Memory,5.0,"The clocks are burnt on a hot desert, it embod...",4 - Agree,3 - Neutral/Unsure,2 - Disagree,1 - Strongly disagree,2.0,4.0,0,Bathroom,By yourself,Fall,Fries,A country song that contrasts nostalgia for th...
2,3,The Persistence of Memory,7.0,This painting makes me feel dread. The clock r...,4 - Agree,1 - Strongly disagree,1 - Strongly disagree,4 - Agree,4.0,3.0,$5,"Bathroom,Dining room","Coworkers/Classmates,By yourself",Fall,Sardines,A melancholy instrumental with a monotone voic...
3,4,The Persistence of Memory,7.0,Deflated,4 - Agree,1 - Strongly disagree,2 - Disagree,4 - Agree,10.0,7.0,a,"Bedroom,Bathroom",Coworkers/Classmates,Winter,a,q
4,5,The Persistence of Memory,7.0,The painting gives me a sense of calmness and ...,3 - Neutral/Unsure,4 - Agree,5 - Strongly agree,3 - Neutral/Unsure,4.0,6.0,300 dollars.,Living room,Friends,"Spring,Summer",Churros.,Radiohead's album in rainbows.
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1681,558,The Water Lily Pond,5.0,Joy and relaxation.,2 - Disagree,4 - Agree,4 - Agree,2 - Disagree,2.0,1.0,$35,"Bedroom,Office,Living room,Dining room","Friends,Family members,Coworkers/Classmates,St...",Spring,green apple,nature forest soundtrack
1682,559,The Water Lily Pond,9.0,This painting makes me feel happy,2 - Disagree,5 - Strongly agree,5 - Strongly agree,1 - Strongly disagree,3.0,3.0,200,"Bedroom,Office,Living room",Friends,"Spring,Summer",Salad,Slow guitar soundtrack
1683,560,The Water Lily Pond,8.0,This painting makes me feel warm and new birth...,1 - Strongly disagree,5 - Strongly agree,3 - Neutral/Unsure,1 - Strongly disagree,2.0,2.0,"200 Dollars, pay for the emotion","Bedroom,Living room","Friends,Family members",Spring,Fresh salad,"Various animal sounds, like bird sounds; and l..."
1684,561,The Water Lily Pond,5.0,makes me feel nonchalant,2 - Disagree,2 - Disagree,4 - Agree,1 - Strongly disagree,4.0,1.0,10,"Bathroom,Office","Friends,Family members,Coworkers/Classmates,St...",Spring,apple,simon and garfunkel


In [200]:
# Renaming Column names

df = df.rename(columns={'unique_id': 'rid',
                        'Painting': 'label', 'On a scale of 1–10, how intense is the emotion conveyed by the artwork?': 'intensity',
                        'Describe how this painting makes you feel.': 'feeling',
                        'This art piece makes me feel sombre.': 'sombre?',
                        'This art piece makes me feel content.': 'content?',
                        'This art piece makes me feel calm.': 'calm?',
                        'This art piece makes me feel uneasy.': 'uneasy?',
                        'How many prominent colours do you notice in this painting?': 'num_colors',
                        'How many objects caught your eye in the painting?': 'num_objects',
                        'How much (in Canadian dollars) would you be willing to pay for this painting?': 'worth',
                        'If you could purchase this painting, which room would you put that painting in?': 'place',
                        'If you could view this art in person, who would you want to view it with?': 'view_with',
                        'What season does this art piece remind you of?': 'season',
                        'If this painting was a food, what would be?': 'food',
                        'Imagine a soundtrack for this painting. Describe that soundtrack without naming any objects in the painting.': 'music'})

In [201]:
# We split the data into training and validation, using the stratified 80/20 split.

# from sklearn.model_selection import train_test_split

# train_df, val_df = train_test_split(
#     df,
#     test_size=0.2,
#     random_state=42,
#     stratify=df["label"]
# )

# We figured that stratified was leaking data so now we try splitting the dataset
# based on the Unique id's

from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, val_idx = next(gss.split(df, groups=df["rid"]))

train_df = df.iloc[train_idx].copy()
val_df = df.iloc[val_idx].copy()


In [202]:
# Adding column class
# Overwritten by Haseeb
class_map = {
    "The Persistence of Memory": 0,
    "The Starry Night": 1,
    "The Water Lily Pond": 2
}

train_df["Class"] = train_df["label"].map(class_map)
val_df["Class"] = val_df["label"].map(class_map)

In [203]:
import re
import numpy as np
import pandas as pd

# Haseeb
# cleaning the worth column

def clean_worth_col(val):
    if pd.isna(val):
        return np.nan

    s = str(val).strip().lower()
    missing_tokens = {
        "", "na", "n/a", "none", "null", "nan",
        "idk", "i dont know", "i don't know",
        "dont know", "don't know", "unknown",
        "priceless", "free"
    }

    if s in missing_tokens:
        return np.nan

    # remove common noise
    s = s.replace(",", "")
    s = s.replace("$", "")
    s = s.replace("cad", "")
    s = s.replace("canadian dollars", "")
    s = s.strip()
    s = s.replace(" ", "")

    # extract numeric tokens
    number = re.findall(r'\d+(?:\.\d+)?', s)

    # handle ranges
    if len(number) >= 2 and ("-" in s or "between" in s):
        return (float(number[0]) + float(number[1])) / 2

    # handle billion / million / k
    if number:
        num = float(number[0])

        if "billion" in s:
            return num * 1_000_000_000

        if "million" in s:
            return num * 1_000_000

        if re.search(r'\bk\b', s) or s.endswith("k"):
            return num * 1_000

        return num

    return np.nan


def fit_worth_preprocessing(train_df):
    # parse raw worth on train
    train_worth_clean = train_df["worth"].apply(clean_worth_col)

    # learn train-only statistics
    worth_median = train_worth_clean.median()

    train_worth_clean = train_worth_clean.fillna(worth_median)
    train_worth_log = np.log1p(train_worth_clean)

    worth_mean = train_worth_log.mean()
    worth_std = train_worth_log.std()

    return worth_median, worth_mean, worth_std


def apply_worth_preprocessing(df, worth_median, worth_mean, worth_std):
    df["worth_clean"] = df["worth"].apply(clean_worth_col)
    df["worth_clean"] = df["worth_clean"].fillna(worth_median)

    df["worth_log"] = np.log1p(df["worth_clean"])
    df["worth_scaled"] = (df["worth_log"] - worth_mean) / worth_std

    return df

worth_median, worth_mean, worth_std = fit_worth_preprocessing(train_df)

train_df = apply_worth_preprocessing(train_df, worth_median, worth_mean, worth_std)
val_df = apply_worth_preprocessing(val_df, worth_median, worth_mean, worth_std)

In [204]:
train_df.drop(columns=["worth", "worth_log"], inplace=True)
val_df.drop(columns=["worth", "worth_log"], inplace=True)

In [205]:
from sklearn.preprocessing import MultiLabelBinarizer

def split_and_clean_multilabel(series):
    return (
        series.fillna("")
        .str.split(",")
        .apply(lambda xs: [x.strip().lower().replace(" ", "_") for x in xs if x.strip()])
    )

# ---------- PLACE ----------
train_df["place_list"] = split_and_clean_multilabel(train_df["place"])
val_df["place_list"] = split_and_clean_multilabel(val_df["place"])

mlb_place = MultiLabelBinarizer()
place_train = mlb_place.fit_transform(train_df["place_list"])
place_val = mlb_place.transform(val_df["place_list"])

place_train_df = pd.DataFrame(
    place_train,
    columns=[f"place_{c}" for c in mlb_place.classes_],
    index=train_df.index
)

place_val_df = pd.DataFrame(
    place_val,
    columns=[f"place_{c}" for c in mlb_place.classes_],
    index=val_df.index
)

train_df = pd.concat([train_df, place_train_df], axis=1)
val_df = pd.concat([val_df, place_val_df], axis=1)

train_df.drop(columns=["place", "place_list"], inplace=True)
val_df.drop(columns=["place", "place_list"], inplace=True)


# ---------- VIEW_WITH ----------
train_df["view_with_list"] = split_and_clean_multilabel(train_df["view_with"])
val_df["view_with_list"] = split_and_clean_multilabel(val_df["view_with"])

mlb_view = MultiLabelBinarizer()
view_train = mlb_view.fit_transform(train_df["view_with_list"])
view_val = mlb_view.transform(val_df["view_with_list"])

view_train_df = pd.DataFrame(
    view_train,
    columns=[f"view_with_{c}" for c in mlb_view.classes_],
    index=train_df.index
)

view_val_df = pd.DataFrame(
    view_val,
    columns=[f"view_with_{c}" for c in mlb_view.classes_],
    index=val_df.index
)

train_df = pd.concat([train_df, view_train_df], axis=1)
val_df = pd.concat([val_df, view_val_df], axis=1)

train_df.drop(columns=["view_with", "view_with_list"], inplace=True)
val_df.drop(columns=["view_with", "view_with_list"], inplace=True)


# ---------- SEASON ----------
train_df["season_list"] = split_and_clean_multilabel(train_df["season"])
val_df["season_list"] = split_and_clean_multilabel(val_df["season"])

mlb_season = MultiLabelBinarizer()
season_train = mlb_season.fit_transform(train_df["season_list"])
season_val = mlb_season.transform(val_df["season_list"])

season_train_df = pd.DataFrame(
    season_train,
    columns=[f"season_{c}" for c in mlb_season.classes_],
    index=train_df.index
)

season_val_df = pd.DataFrame(
    season_val,
    columns=[f"season_{c}" for c in mlb_season.classes_],
    index=val_df.index
)

train_df = pd.concat([train_df, season_train_df], axis=1)
val_df = pd.concat([val_df, season_val_df], axis=1)

train_df.drop(columns=["season", "season_list"], inplace=True)
val_df.drop(columns=["season", "season_list"], inplace=True)

In [206]:
# Aarushi

# numeric columns: fit medians on train, apply to both
intensity_median = train_df['intensity'].median()
num_colors_median = train_df['num_colors'].median()
num_objects_median = train_df['num_objects'].median()

train_df['intensity'] = train_df['intensity'].fillna(intensity_median)
val_df['intensity'] = val_df['intensity'].fillna(intensity_median)

train_df['num_colors'] = train_df['num_colors'].fillna(num_colors_median)
val_df['num_colors'] = val_df['num_colors'].fillna(num_colors_median)

train_df['num_objects'] = train_df['num_objects'].fillna(num_objects_median)
val_df['num_objects'] = val_df['num_objects'].fillna(num_objects_median)

# Likert-style columns: extract digits in both, fit medians on train, apply to both
cols = ['sombre?', 'content?', 'calm?', 'uneasy?']

for col in cols:
    train_df[col] = train_df[col].astype(str).str.extract(r'(\d+)').astype(float)
    val_df[col] = val_df[col].astype(str).str.extract(r'(\d+)').astype(float)

for col in cols:
    col_median = train_df[col].median()
    train_df[col] = train_df[col].fillna(col_median)
    val_df[col] = val_df[col].fillna(col_median)

In [207]:
# Cleaning the feeling column and applying TF-IDF vectorization to it.

# from sklearn.feature_extraction.text import TfidfVectorizer
# from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

# custom_stops = [
#     'feel', 'feeling', 'feels', 'gives', 'makes',
#     'make', 'like', 'bit', 'sense', 'world',
#     'painting', 'little', 'life', 'away', 'look',
#     'looking', 'really', 'reminds', 'think', 'way', 'wonder', 'want'
# ]

# vectorizer = TfidfVectorizer(
#     max_features=50,
#     stop_words=list(ENGLISH_STOP_WORDS) + custom_stops
# )

# feeling_tfidf_train = vectorizer.fit_transform(train_df['feeling'].fillna(''))

# feeling_train_df = pd.DataFrame(
#     feeling_tfidf_train.toarray(),
#     columns=vectorizer.get_feature_names_out(),
#     index=train_df.index
# )

# train_df = pd.concat([train_df, feeling_train_df], axis=1)
# train_df = train_df.drop(columns=['feeling'])

# feeling_tfidf_val = vectorizer.transform(val_df['feeling'].fillna(''))

# feeling_val_df = pd.DataFrame(
#     feeling_tfidf_val.toarray(),
#     columns=vectorizer.get_feature_names_out(),
#     index=val_df.index
# )

# val_df = pd.concat([val_df, feeling_val_df], axis=1)
# val_df = val_df.drop(columns=['feeling'])

# BAG OF WORDS (UNIGRAM)

from collections import Counter
import re

custom_stops = {
    'feel', 'feeling', 'feels', 'gives', 'makes',
    'make', 'like', 'bit', 'sense', 'world',
    'painting', 'little', 'life', 'away', 'look',
    'looking', 'really', 'reminds', 'think', 'way', 'wonder', 'want',
    'the', 'a', 'an', 'and', 'or', 'to', 'of', 'in', 'on', 'at', 'for', 'with', 'by',
    'is', 'it', 'this', 'that', 'as', 'be', 'are', 'was', 'were',
    'i', 'me', 'my', 'we', 'our', 'you', 'your', 'there', 'also', 'can', 'if', 'but', 'about'
}

def tokenize_unigram(text):
    if pd.isna(text):
        return []

    text = str(text).lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    if text == "":
        return []

    tokens = text.split()
    tokens = [tok for tok in tokens if tok not in custom_stops and len(tok) > 1]
    return tokens

def build_vocab(text_series, min_df=2, max_features=50):
    doc_freq = Counter()

    for text in text_series.fillna(""):
        unique_tokens = set(tokenize_unigram(text))
        doc_freq.update(unique_tokens)

    vocab_tokens = [tok for tok, freq in doc_freq.items() if freq >= min_df]
    vocab_tokens.sort(key=lambda tok: (-doc_freq[tok], tok))
    vocab_tokens = vocab_tokens[:max_features]

    return vocab_tokens

def transform_binary_bow(text_series, vocab):
    bow_rows = []

    for text in text_series.fillna(""):
        tokens = set(tokenize_unigram(text))
        row = {f"feeling_word_{word}": int(word in tokens) for word in vocab}
        bow_rows.append(row)

    return pd.DataFrame(bow_rows, index=text_series.index)

# Build vocab from TRAINING ONLY
feeling_vocab = build_vocab(train_df["feeling"], min_df=2, max_features=50)

# Freeze vocab and apply to both train and val
feeling_train_df = transform_binary_bow(train_df["feeling"], feeling_vocab)
feeling_val_df = transform_binary_bow(val_df["feeling"], feeling_vocab)

train_df = pd.concat([train_df, feeling_train_df], axis=1)
val_df = pd.concat([val_df, feeling_val_df], axis=1)

train_df = train_df.drop(columns=["feeling"])
val_df = val_df.drop(columns=["feeling"])

In [208]:
# Linear Regression - Sid

from sklearn.linear_model import LinearRegression
from sklearn.metrics import accuracy_score
import numpy as np
import pandas as pd

lr_results = []

for group_name in feature_groups:
    X_train = X_train_groups[group_name]
    X_val = X_val_groups[group_name]

    model = LinearRegression()
    model.fit(X_train, y_train)

    # Predict continuous values and round to nearest class (0, 1, 2)
    train_pred = np.clip(np.round(model.predict(X_train)).astype(int), 0, 2)
    val_pred   = np.clip(np.round(model.predict(X_val)).astype(int), 0, 2)

    train_acc = accuracy_score(y_train, train_pred)
    val_acc   = accuracy_score(y_val,   val_pred)

    lr_results.append({
        "feature_group": group_name,
        "num_features":  X_train.shape[1],
        "train_accuracy": train_acc,
        "val_accuracy":   val_acc
    })

    print(f"=== {group_name} ===")
    print(f"Number of features: {X_train.shape[1]}")
    print(f"Train Accuracy: {train_acc:.4f}")
    print(f"Validation Accuracy: {val_acc:.4f}")
    print()

lr_results_df = pd.DataFrame(lr_results).sort_values("val_accuracy", ascending=False)
print("Linear Regression Results:")
print(lr_results_df)

=== A_numeric_only ===
Number of features: 8
Train Accuracy: 0.7112
Validation Accuracy: 0.7316

=== B_numeric_no_worth ===
Number of features: 7
Train Accuracy: 0.7120
Validation Accuracy: 0.7345

=== C_text_only ===
Number of features: 50
Train Accuracy: 0.6318
Validation Accuracy: 0.5959

=== D_categorical_only ===
Number of features: 14
Train Accuracy: 0.6741
Validation Accuracy: 0.6755

=== E_all_except_food_music ===
Number of features: 72
Train Accuracy: 0.8137
Validation Accuracy: 0.7906

Linear Regression Results:
             feature_group  num_features  train_accuracy  val_accuracy
4  E_all_except_food_music            72        0.813660      0.790560
1       B_numeric_no_worth             7        0.711952      0.734513
0           A_numeric_only             8        0.711210      0.731563
3       D_categorical_only            14        0.674091      0.675516
2              C_text_only            50        0.631774      0.595870


In [209]:
train_df[['feeling_word_happy', 'label']].iloc[100:120]

,feeling_word_happy,label
133,0,The Persistence of Memory
134,0,The Persistence of Memory
135,0,The Persistence of Memory
136,0,The Persistence of Memory
138,0,The Persistence of Memory
139,0,The Persistence of Memory
141,0,The Persistence of Memory
142,0,The Persistence of Memory
143,0,The Persistence of Memory
146,0,The Persistence of Memory


In [210]:
feeling_cols = [col for col in train_df.columns if col.startswith("feelingword")]

train_df_temp = train_df[feeling_cols + ['label']].copy()
print(train_df_temp.groupby('label').mean().T)

Empty DataFrame
Columns: [The Persistence of Memory, The Starry Night, The Water Lily Pond]
Index: []


In [211]:
print(train_df.iloc[0].to_string())
#print(val_df.iloc[0].to_string())

rid                                                       1
label                             The Persistence of Memory
intensity                                               7.0
sombre?                                                 3.0
content?                                                4.0
calm?                                                   4.0
uneasy?                                                 2.0
num_colors                                              5.0
num_objects                                             3.0
food                                                    NaN
music                                                   NaN
Class                                                     0
worth_clean                                           100.0
worth_scaled                                      -0.315023
place_bathroom                                            0
place_bedroom                                             0
place_dining_room                       

In [212]:
# Haseeb

# * Ok so now we will start with creating subsets of our features, then train the model on each of them
# * We are starting with ommiting the music and food column

# CONSTRUCTING SUBSET OF FEATURES:

# Group A:
# Numerical only, so all features that have a numerical value

# Group B:
# All numerical features/col without worth because I feel most people answering the questionaire
# don't know how paintings are priced... so their worth inputs might confuse the model
# and create noise. This is my hypothesis, which I want to validate through training + validation runs

# Group C:
# Text only columns, so basically just the feeling feature/column

# Group D:
# Categorical features/cols only so place, view_with and season

# Group E
# All features except Music and Food since we haven't yet been able to figure
# how to preprocess them.


# Group A = Numerical / ordinal features
numeric_cols = [
    "intensity",
    "sombre?",
    "content?",
    "calm?",
    "uneasy?",
    "num_colors",
    "num_objects",
    "worth_scaled"
]

# Group B = numerical without worth
numeric_no_worth_cols = [col for col in numeric_cols if col != "worth_scaled"]

# Group C = Text-only features
# These are the TF-IDF columns created from the feeling column
text_cols = [col for col in train_df.columns if col.startswith("feeling_word_")]

# Group D = Categorical-only features
# These came from place / view_with / season one-hot encoding
categorical_cols = [
    col for col in train_df.columns
    if col.startswith("place_") or col.startswith("view_with_") or col.startswith("season_")
]


# Group E = All usable features except food/music
all_feature_cols = numeric_cols + text_cols + categorical_cols

# Remove accidental duplicates just in case
all_feature_cols = list(dict.fromkeys(all_feature_cols))

feature_groups = {
    "A_numeric_only": numeric_cols,
    "B_numeric_no_worth": numeric_no_worth_cols,
    "C_text_only": text_cols,
    "D_categorical_only": categorical_cols,
    "E_all_except_food_music": all_feature_cols
}

In [213]:
# We seperate the train and val groups:
X_train_groups = {}
X_val_groups = {}

for group_name, cols in feature_groups.items():
    X_train_groups[group_name] = train_df[cols].copy()
    X_val_groups[group_name] = val_df[cols].copy()

# targets
y_train = train_df["Class"]
y_val = val_df["Class"]

In [214]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
import pandas as pd

results = []

for group_name in feature_groups:
    X_train = X_train_groups[group_name]
    X_val = X_val_groups[group_name]

    model = LogisticRegression(max_iter=2000, random_state=42)
    model.fit(X_train, y_train)

    train_pred = model.predict(X_train)
    val_pred = model.predict(X_val)

    train_acc = accuracy_score(y_train, train_pred)
    val_acc = accuracy_score(y_val, val_pred)

    results.append({
        "feature_group": group_name,
        "num_features": X_train.shape[1],
        "train_accuracy": train_acc,
        "val_accuracy": val_acc
    })

    print(f"=== {group_name} ===")
    print(f"Number of features: {X_train.shape[1]}")
    print(f"Train Accuracy: {train_acc:.4f}")
    print(f"Validation Accuracy: {val_acc:.4f}")
    print()

results_df = pd.DataFrame(results).sort_values("val_accuracy", ascending=False)
print(results_df)

=== A_numeric_only ===
Number of features: 8
Train Accuracy: 0.7498
Validation Accuracy: 0.7670

=== B_numeric_no_worth ===
Number of features: 7
Train Accuracy: 0.7379
Validation Accuracy: 0.7611

=== C_text_only ===
Number of features: 50
Train Accuracy: 0.7008
Validation Accuracy: 0.6903

=== D_categorical_only ===
Number of features: 14
Train Accuracy: 0.7743
Validation Accuracy: 0.7847

=== E_all_except_food_music ===
Number of features: 72
Train Accuracy: 0.9050
Validation Accuracy: 0.8555

             feature_group  num_features  train_accuracy  val_accuracy
4  E_all_except_food_music            72        0.904974      0.855457
3       D_categorical_only            14        0.774313      0.784661
0           A_numeric_only             8        0.749814      0.766962
1       B_numeric_no_worth             7        0.737936      0.761062
2              C_text_only            50        0.700817      0.690265
